# **Daily Aggregation (Silver → Daily)**

In [0]:
# Notebook 1: Daily Aggregation (Silver → Daily Delta Table)
from pyspark.sql.functions import sum, avg, count, col, when
from pyspark.sql.types import NumericType, StringType
import pandas as pd

df_sales = spark.table("silver_sales_clean")
print("Total rows:", df_sales.count())

# Numeric columns auto-detect
numeric_cols = []
for field in df_sales.schema.fields:
    if isinstance(field.dataType, NumericType) and field.name not in ['order_id']:
        numeric_cols.append(field.name)

categorical_cols = ['zone', 'category', 'brand_type', 'customer_gender', 
                    'sales_event', 'competition_intensity', 'inventory_pressure']

# Numeric aggregations
agg_exprs = []
for c in numeric_cols:
    if c in ['units_sold', 'revenue']:
        agg_exprs.append(sum(c).alias(f"total_{c}"))
    else:
        agg_exprs.append(avg(c).alias(f"avg_{c}"))
agg_exprs.append(count("order_id").alias("order_count"))

daily_numeric = df_sales.groupBy("order_date").agg(*agg_exprs)

# Categorical proportions
daily_categorical = None
for cat_col in categorical_cols:
    distinct_vals = df_sales.select(cat_col).distinct().toPandas()[cat_col].tolist()
    for val in distinct_vals:
        col_name = f"pct_{cat_col}_{val}".replace(" ", "_").replace("&", "").replace("(", "").replace(")", "").lower()
        prop_expr = avg(when(col(cat_col) == val, 1).otherwise(0)).alias(col_name)
        temp_df = df_sales.groupBy("order_date").agg(prop_expr)
        if daily_categorical is None:
            daily_categorical = temp_df
        else:
            daily_categorical = daily_categorical.join(temp_df, on="order_date", how="outer")
daily_categorical = daily_categorical.fillna(0)

# Merge
daily = daily_numeric.join(daily_categorical, on="order_date", how="left")

# Save as Delta table (not CSV)
daily.write.mode("overwrite").saveAsTable("daily_aggregated_sales")
print("✅ Daily aggregated data saved as 'daily_aggregated_sales' table")

Total rows: 30600
✅ Daily aggregated data saved as 'daily_aggregated_sales' table
